# Benchmark — Vanilla vs Fine-Tuned: SAM & MedSAM (+ U-Net specialist)
**Project:** Can LoRA-Adapted SAM Match Specialist Polyp Networks? A Cross-Dataset Generalization Study in Colonoscopy

Evaluates **five** models side by side on all 5 splits — 2 seen (Kvasir, CVC-ClinicDB),
3 unseen (CVC-ColonDB, ETIS-LaribPolypDB, CVC-300):

- **U-Net (ResNet-34)** — from-scratch specialist baseline
- **SAM-ViT-H + LoRA**, **MedSAM-ViT-B + LoRA** — *fine-tuned* foundation models
- **SAM-ViT-H (zero-shot)**, **MedSAM-ViT-B (zero-shot)** — *vanilla* foundation models prompted by
  the GT box (the "without fine-tuning" baselines; see `docs/PROJECT_PLAN.md`)

**Requires:** trained `best.pt` for `{unet, sam_vit_h, medsam}` at
`/content/drive/MyDrive/msu2026_checkpoints/<model>/seed42/best.pt` (written by `train.py` /
`train_colab.ipynb`). The zero-shot models need no checkpoint — only the base SAM/MedSAM weights,
downloaded below.

## 1. Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO = '/content/msu2026summer_final_project'
    if not os.path.exists(REPO):
        !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
    %cd {REPO}
    !git fetch --quiet origin && git reset --hard origin/main
    !find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
    !pip install -q -r requirements.txt
    !pip install -q git+https://github.com/facebookresearch/segment-anything.git

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import gdown, shutil, zipfile
from pathlib import Path

DATA_ROOT = Path('data/polyp')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def _dl(file_id, zip_name):
    target = DATA_ROOT / zip_name.replace('.zip', '')
    if target.exists():
        print(f'{target.name}: already present'); return
    zip_path = f'/tmp/{zip_name}'
    print(f'Downloading {target.name}...')
    gdown.download(id=file_id, output=zip_path, quiet=False, fuzzy=True)
    tmp = Path(f'/tmp/extract_{target.name}')
    tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(tmp)
    extracted = [p for p in tmp.iterdir() if not p.name.startswith('__')]
    if len(extracted) == 1 and extracted[0].is_dir():
        shutil.move(str(extracted[0]), str(target))
    else:
        target.mkdir(exist_ok=True)
        for p in extracted: shutil.move(str(p), str(target / p.name))
    print(f'Done -> {target}')

_dl('1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb', 'TrainDataset.zip')
_dl('1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao', 'TestDataset.zip')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "N/A"}')

## 2. Locate Checkpoints

Pulls each model's `best.pt` from Drive into the local `checkpoints/` tree
(matching the path each training notebook used).

In [ ]:
SEED = 42
DRIVE_ROOT = Path('/content/drive/MyDrive/msu2026_checkpoints')

CKPT_MAP = {
    'unet':    Path(f'checkpoints/unet/seed{SEED}/best.pt'),
    'sam_vit_h': Path(f'checkpoints/sam_vit_h/seed{SEED}/best.pt'),
    'medsam':  Path(f'checkpoints/medsam/seed{SEED}/best.pt'),
}

# Optional ablation: SAM ViT-B + LoRA. Included ONLY if it was trained (sam_b in train_colab);
# absent by default, so the benchmark never fails just because the ablation wasn't run.
OPTIONAL_CKPTS = {'sam_vit_b': Path(f'checkpoints/sam_vit_b/seed{SEED}/best.pt')}

for name, local_path in CKPT_MAP.items():
    if local_path.exists():
        print(f'{name:<12} found locally  -> {local_path}')
        continue
    drive_path = DRIVE_ROOT / name / f'seed{SEED}' / 'best.pt'
    if not drive_path.exists():
        raise FileNotFoundError(
            f'No checkpoint for "{name}" at {drive_path}. '
            f'Run the corresponding training notebook and its '
            f'"Save Checkpoint to Google Drive" cell first.'
        )
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(drive_path, local_path)
    print(f'{name:<12} pulled from Drive -> {local_path}')

AVAILABLE_OPTIONAL = {}
for name, local_path in OPTIONAL_CKPTS.items():
    if not local_path.exists():
        drive_path = DRIVE_ROOT / name / f'seed{SEED}' / 'best.pt'
        if drive_path.exists():
            local_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(drive_path, local_path)
    if local_path.exists():
        AVAILABLE_OPTIONAL[name] = local_path
        print(f'{name:<12} found (optional) -> {local_path}')
    else:
        print(f'{name:<12} not trained yet (optional ablation) -> skipping')

## 3. Build and Load All Models (fine-tuned + zero-shot)

In [ ]:
from src.models import build_unet, build_sam_lora, build_medsam_lora
from src.models.unet import count_parameters

IMG_SIZE   = 352
LORA_R     = 4
LORA_ALPHA = 8.0

models = {}

# --- U-Net ---
unet = build_unet().to(device)
unet.load_state_dict(torch.load(CKPT_MAP['unet'], map_location=device))
unet.eval()
models['U-Net (ResNet-34)'] = unet

# --- SAM ViT-H + LoRA ---
SAM_FNAME = 'sam_vit_h_4b8939.pth'
if not os.path.exists(SAM_FNAME):
    print('Downloading SAM ViT-H checkpoint (~2.4 GB, needed to rebuild the architecture)...')
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O {SAM_FNAME}

sam_lora = build_sam_lora(
    sam_checkpoint=SAM_FNAME, model_type='vit_h',
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1,
    img_size=IMG_SIZE, device=device,
).to(device)
sam_lora.load_state_dict(torch.load(CKPT_MAP['sam_vit_h'], map_location=device))
sam_lora.eval()
models['SAM-ViT-H + LoRA'] = sam_lora

# --- MedSAM + LoRA ---
import hashlib
MEDSAM_FNAME = 'medsam_vit_b.pth'
MEDSAM_URL   = 'https://zenodo.org/records/10689643/files/medsam_vit_b.pth'
MEDSAM_MD5   = '3bb6db55bd0c9ca30b61248bca72f8d6'

def _md5(path, chunk=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

if os.path.exists(MEDSAM_FNAME) and (os.path.getsize(MEDSAM_FNAME) < 300_000_000 or _md5(MEDSAM_FNAME) != MEDSAM_MD5):
    os.remove(MEDSAM_FNAME)
if not os.path.exists(MEDSAM_FNAME):
    print('Downloading MedSAM checkpoint (~375 MB, needed to rebuild the architecture)...')
    !wget -q --show-progress {MEDSAM_URL} -O {MEDSAM_FNAME}
assert _md5(MEDSAM_FNAME) == MEDSAM_MD5, 'MedSAM checkpoint MD5 mismatch'

medsam_lora = build_medsam_lora(
    medsam_checkpoint=MEDSAM_FNAME,
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1,
    img_size=IMG_SIZE, device=device,
).to(device)
medsam_lora.load_state_dict(torch.load(CKPT_MAP['medsam'], map_location=device))
medsam_lora.eval()
models['MedSAM-ViT-B + LoRA'] = medsam_lora

# --- SAM ViT-B + LoRA (optional ablation) ---
# Same ViT-B backbone as MedSAM but with generic SAM weights, so comparing this row to
# MedSAM-ViT-B separates "backbone capacity" from "medical pretraining".
if 'sam_vit_b' in AVAILABLE_OPTIONAL:
    SAM_B_FNAME = 'sam_vit_b_01ec64.pth'
    if not os.path.exists(SAM_B_FNAME):
        print('Downloading SAM ViT-B checkpoint (~375 MB, needed to rebuild the architecture)...')
        !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {SAM_B_FNAME}
    sam_b_lora = build_sam_lora(
        sam_checkpoint=SAM_B_FNAME, model_type='vit_b',
        lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1,
        img_size=IMG_SIZE, device=device,
    ).to(device)
    sam_b_lora.load_state_dict(torch.load(AVAILABLE_OPTIONAL['sam_vit_b'], map_location=device))
    sam_b_lora.eval()
    models['SAM-ViT-B + LoRA'] = sam_b_lora

# --- Zero-shot (vanilla, un-fine-tuned) baselines: the "without fine-tuning" half ---
# Reuse the SAM / MedSAM checkpoints downloaded above; no training, no LoRA, 0 trainable params.
# Prompted by the GT bounding box (oracle prompt). Swap ZS_PROMPT / ZS_BOX_PAD to change protocol
# (see docs/PROJECT_PLAN.md). Note: zero-shot SAM ViT-H adds ~2.5 GB VRAM on top of SAM-LoRA — L4/A100.
from src.models import build_zeroshot_sam, build_zeroshot_medsam

ZS_PROMPT  = 'box'   # 'box' (recommended, oracle box from GT) | 'point'
ZS_BOX_PAD = 5

zeroshot_models = {}
zeroshot_models['SAM-ViT-H (zero-shot)'] = build_zeroshot_sam(
    SAM_FNAME, model_type='vit_h', device=device, prompt=ZS_PROMPT, box_padding=ZS_BOX_PAD)
zeroshot_models['MedSAM-ViT-B (zero-shot)'] = build_zeroshot_medsam(
    MEDSAM_FNAME, device=device, prompt=ZS_PROMPT, box_padding=ZS_BOX_PAD)

# Unified dict every table/plot below iterates over: fine-tuned + vanilla + U-Net reference.
all_models = {**models, **zeroshot_models}
print('Fine-tuned / trained models:', list(models.keys()))
print('Zero-shot baselines        :', list(zeroshot_models.keys()))

## 4. Parameter Efficiency Table

How many parameters each method actually trains, relative to its total size.

In [ ]:
rows = []
for name, m in all_models.items():
    if hasattr(m, 'total_parameters'):
        total, trainable = m.total_parameters(), m.trainable_parameters()
    else:
        p = count_parameters(m)
        total, trainable = p['total'], p['trainable']
    rows.append((name, total, trainable))

print(f'{"Method":<26} {"Total params":>14}  {"Trainable":>14}  {"% trainable":>12}')
print('-' * 74)
for name, total, trainable in rows:
    print(f'{name:<26} {total:>14,}  {trainable:>14,}  {100*trainable/total:>11.2f}%')

## 5. Full Evaluation — All Models × All Splits

Same protocol used in notebooks 02-04: mDice, mIoU, MAE, weighted Fβ, S-measure, E-measure.

In [ ]:
from torch.utils.data import DataLoader
from src.data import PolypDataset, build_splits, get_val_transform
from src.metrics import MetricTracker
from src.models import is_zeroshot, predict_zeroshot_prob

splits = build_splits(DATA_ROOT, seed=SEED)
val_transform = get_val_transform(IMG_SIZE)

split_labels = {
    'Seen — Kvasir':        'seen_kvasir',
    'Seen — CVC-ClinicDB':  'seen_clinicdb',
    'Unseen — CVC-ColonDB': 'cvc_colondb',
    'Unseen — ETIS-Larib':  'etis_larib',
    'Unseen — CVC-300':     'cvc_300',
}

all_results = {}  # all_results[model_name][split_key] = {dice, iou, mae, wfm, sm, em}

for model_name, model in all_models.items():
    print(f'\n=== {model_name} ===')
    print(f'{"Split":<26} {"mDice":>7} {"mIoU":>7} {"MAE":>7} {"wFm":>7} {"Sm":>7} {"Em":>7}')
    print('-' * 75)
    all_results[model_name] = {}
    for label, key in split_labels.items():
        if key not in splits:
            print(f'{label:<26} -- not in splits --')
            continue
        tracker = MetricTracker()
        if is_zeroshot(model):
            # Vanilla SAM/MedSAM: prompt from the GT mask; feed raw uint8 images (SAM does its own
            # normalisation), evaluated at IMG_SIZE so metrics match the fine-tuned models.
            for ip, mp in zip(splits[key]['image_paths'], splits[key]['mask_paths']):
                prob, gt = predict_zeroshot_prob(model, ip, mp, IMG_SIZE)
                tracker.update(prob, gt)
        else:
            ds = PolypDataset(splits[key]['image_paths'], splits[key]['mask_paths'], transform=val_transform)
            loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=2)
            with torch.no_grad():
                for imgs, masks in loader:
                    probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
                    for i in range(len(probs)):
                        tracker.update(probs[i, 0], masks[i, 0].numpy())
        sc = tracker.compute()
        all_results[model_name][key] = sc
        print(f'{label:<26} {sc["dice"]:>7.4f} {sc["iou"]:>7.4f} {sc["mae"]:>7.4f} '
              f'{sc["wfm"]:>7.4f} {sc["sm"]:>7.4f} {sc["em"]:>7.4f}')

## 6. Comparison Table — mDice per Split

In [ ]:
import pandas as pd

split_keys = list(split_labels.values())
table = pd.DataFrame({
    model_name: [all_results[model_name].get(k, {}).get('dice', float('nan')) for k in split_keys]
    for model_name in all_models
}, index=list(split_labels.keys()))

table.loc['Mean — Seen']   = table.loc[['Seen — Kvasir', 'Seen — CVC-ClinicDB']].mean()
table.loc['Mean — Unseen'] = table.loc[['Unseen — CVC-ColonDB', 'Unseen — ETIS-Larib', 'Unseen — CVC-300']].mean()
table.loc['Generalization Gap (Seen − Unseen)'] = table.loc['Mean — Seen'] - table.loc['Mean — Unseen']

table.round(4)

## 6b. Multi-Seed Reliability — Mean ± Std Across Seeds

`train_colab.ipynb` cell 1 trains every model over `SEEDS` (see `configs/base.yaml`'s `n_runs: 3`),
writing one `results/<model>/seed<seed>/metrics.json` per run (schema: `src/training/reporting.py`
`build_metrics_payload`). This aggregates those per-seed files into mean ± std mDice per split for
the **prompt-free** trained models only — U-Net, SAM+LoRA, MedSAM+LoRA. The oracle-prompted
zero-shot baselines are untrained (0 trainable params, no seed dependence), so they're excluded here.

Reads local `results/...` first, falling back to the Drive mirror; a model/seed with no
`metrics.json` yet is simply skipped. Std is reported as `0.0` (not NaN) when only one seed's
results are available — re-run once more seeds land.

In [ ]:
# Aggregate per-seed metrics.json (written by train.py via src/training/reporting.py) into
# mean ± std mDice per split, for the prompt-free trained models only.
import json
import numpy as np

SEEDS = [42, 43, 44]  # [TWEAK] keep in sync with train_colab.ipynb cell 1's SEEDS
# checkpoint/results directory name -> display name (matches CKPT_MAP's keys above)
PROMPT_FREE_DIRS = {'unet': 'U-Net (ResNet-34)', 'sam_vit_h': 'SAM-ViT-H + LoRA',
                    'sam_vit_b': 'SAM-ViT-B + LoRA', 'medsam': 'MedSAM-ViT-B + LoRA'}


def _load_seed_metrics(dir_name, seeds):
    """Load metrics.json for each seed that has one locally or on Drive; keyed by seed."""
    per_seed = {}
    for s in seeds:
        candidates = [
            Path(f'results/{dir_name}/seed{s}/metrics.json'),
            DRIVE_ROOT / 'results' / dir_name / f'seed{s}' / 'metrics.json',
        ]
        for c in candidates:
            if c.exists():
                per_seed[s] = json.loads(c.read_text())
                break
    return per_seed


agg_rows = []
for dir_name, display_name in PROMPT_FREE_DIRS.items():
    per_seed = _load_seed_metrics(dir_name, SEEDS)
    row = {'model': display_name, 'n_seeds': len(per_seed), 'seeds': sorted(per_seed) or None}
    if not per_seed:
        agg_rows.append(row)
        continue
    for label, key in split_labels.items():
        vals = [m['eval'][key]['dice'] for m in per_seed.values() if key in m.get('eval', {})]
        if not vals:
            continue
        row[f'{label} mean'] = float(np.mean(vals))
        row[f'{label} std'] = float(np.std(vals, ddof=0)) if len(vals) > 1 else 0.0
    val_dice = [m['best_val_dice'] for m in per_seed.values()]
    row['best_val_dice mean'] = float(np.mean(val_dice))
    row['best_val_dice std'] = float(np.std(val_dice, ddof=0)) if len(val_dice) > 1 else 0.0
    agg_rows.append(row)

agg_table = pd.DataFrame(agg_rows).set_index('model')
print('Multi-seed aggregation (mean ± std mDice across available seeds per model; '
      'std = 0.0 for a single seed):')
agg_table.round(4)

## 7. Seen vs Unseen — Bar Chart

Visualizes the generalization gap for each method.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

method_names = list(all_models.keys())
seen_vals   = [table.loc['Mean — Seen', m] for m in method_names]
unseen_vals = [table.loc['Mean — Unseen', m] for m in method_names]

x = np.arange(len(method_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, seen_vals,   width, label='Seen (Kvasir + ClinicDB)')
ax.bar(x + width/2, unseen_vals, width, label='Unseen (ColonDB + ETIS + CVC-300)')
ax.set_ylabel('mDice')
ax.set_title('Seen vs Unseen Performance — vanilla vs fine-tuned')
ax.set_xticks(x)
ax.set_xticklabels(method_names, rotation=25, ha='right')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Parameter Efficiency vs Performance

Trainable parameters (log scale) vs mean unseen Dice — the core thesis question.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, total, trainable in rows:
    if name not in table.columns:
        continue
    unseen_dice = table.loc['Mean — Unseen', name]
    zs = (trainable == 0)
    x = max(trainable, 1)  # log axis can't show 0; zero-shot (0 trainable) is drawn at x=1
    ax.scatter(x, unseen_dice, s=150,
               marker='D' if zs else 'o',
               facecolors='none' if zs else 'C0',
               edgecolors='C3' if zs else 'C0',
               linewidths=1.6, zorder=3)
    label = f'{name}\n(0 trainable / zero-shot)' if zs else name
    ax.annotate(label, (x, unseen_dice), textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Trainable Parameters (log scale; zero-shot = 0, plotted at 1)')
ax.set_ylabel('Mean Unseen mDice')
ax.set_title('Parameter Efficiency vs Cross-Dataset Generalization')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Qualitative Comparison

Same unseen test images, all three models' predictions side by side.

In [ ]:
from PIL import Image as PILImage
from src.models import is_zeroshot

def show_comparison(models_dict, image_paths, mask_paths, n=4, img_size=352):
    transform = get_val_transform(img_size)
    n = min(n, len(image_paths))
    fig, axes = plt.subplots(n, 2 + len(models_dict), figsize=(3 * (2 + len(models_dict)), 3 * n))
    if n == 1:
        axes = axes[None, :]

    for row in range(n):
        img_pil  = PILImage.open(image_paths[row]).convert('RGB')
        mask_pil = PILImage.open(mask_paths[row]).convert('L')
        img_np   = np.array(img_pil)
        gt_np    = (np.array(mask_pil) > 127).astype(np.float32)

        aug = transform(image=img_np, mask=gt_np)
        img_t, gt_t = aug['image'].unsqueeze(0).to(device), aug['mask'].numpy()
        img_u8 = np.array(img_pil.resize((img_size, img_size)), dtype=np.uint8)

        axes[row, 0].imshow(img_pil.resize((img_size, img_size)))
        axes[row, 0].set_title('Image' if row == 0 else '')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(gt_t, cmap='gray')
        axes[row, 1].set_title('Ground Truth' if row == 0 else '')
        axes[row, 1].axis('off')

        for col, (name, model) in enumerate(models_dict.items(), start=2):
            if is_zeroshot(model):
                pred = model.predict_prob(img_u8, gt_t)          # GT-box prompted
            else:
                with torch.no_grad():
                    pred = torch.sigmoid(model(img_t)).cpu().numpy()[0, 0]
            pred_bin = (pred >= 0.5).astype(np.float32)
            d = dice_score(pred_bin, gt_t)
            axes[row, col].imshow(pred_bin, cmap='gray')
            axes[row, col].set_title(f'{name}\nDice={d:.3f}' if row == 0 else f'Dice={d:.3f}')
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()

from src.metrics import dice_score

show_comparison(
    all_models,
    splits['cvc_colondb']['image_paths'],
    splits['cvc_colondb']['mask_paths'],
    n=4,
)

## Summary

| Deliverable | Status |
|---|---|
| Trained checkpoints (U-Net, SAM-LoRA, MedSAM-LoRA) loaded | ✅ |
| Zero-shot vanilla SAM + MedSAM baselines (GT-box prompted) | ✅ |
| Parameter / size table (all 5 models) | ✅ |
| Full 5-split evaluation, all models | ✅ |
| mDice comparison table + generalization gap | ✅ |
| Seen vs unseen bar chart | ✅ |
| Parameter efficiency vs performance scatter | ✅ |
| Qualitative side-by-side predictions | ✅ |

**Thesis question:** does fine-tuning (LoRA, ~0.1–3% trainable) beat the *vanilla* foundation
model, and can it match the fully-trained U-Net specialist on **unseen** data? Read two things
together: the **generalization gap** row (seen − unseen) and each model's **mean-unseen mDice** vs
its **trainable params / training time** — that is the accuracy-vs-cost story. The vanilla
SAM/MedSAM rows are GT-box **oracle-prompted** (told roughly where the polyp is) while the
fine-tuned models get no hint — state that caveat when comparing.